In [72]:
"""
s30338, 5.05.2026, generowanie plikow FASTA na podstawie inputu uzytkownika
"""
import os
from os import stat
import random
import re
from typing import List,Tuple

nucleotides = ["A", "C", "G", "T"]

def generate_sequence(length: int, wagi:dict = None) -> str:
    """Zwraca losową sekwencję DNA o zadanej długości."""
    if wagi:#jesli zostaly wprowadzone ilosci % docelowych nukleotydow
      probabilities = [wagi[n] for n in nucleotides]
      return "".join(random.choices(nucleotides, weights=probabilities, k=length))
    return "".join(random.choices(nucleotides, k=length))

def calculate_stats(sequence: str) -> dict:
    """Zwraca słownik ze statystykami sekwencji.
    Klucze: "A", "C", "G", "T" (wartości float, %),
           "GC" (wartość float, %)."""
    length = len(sequence)
    counts = {nucleotyde: sequence.count(nucleotyde) for nucleotyde in nucleotides}
    stats = {}
    for nucleotyde in nucleotides:
        stats[nucleotyde] = round(counts[nucleotyde] / length * 100,2)#obliczenie % dla kazdej z A,C,T,G

    gc = ((counts["G"] + counts["C"]) / length) * 100 #obliczenie % dla GC obok siebie
    stats["GC_ratio"] = round(gc,2)

    return stats

def insert_name(sequence: str, name: str) -> str:
    """Wstawia imię w losową pozycję sekwencji.
    Imię zapisane małymi literami."""
    position = random.randint(0, len(sequence))
    return sequence[:position] + name.lower() + sequence[position:]#wstawienie w srodek imienia napisanego malymi literami

def format_fasta(seq_id: str, description: str,
                 sequence: str, line_width: int = 80) -> str:
    """Zwraca sformatowany rekord FASTA jako string."""
    header = f">{seq_id} "
    if description:
        header += f"{description}"

    lines = [sequence[i:i + line_width] for i in range(0, len(sequence), line_width)] #przerwa co 80 znakow
    return header+"\n" + "\n".join(lines)

def validate_positive_int(prompt: str,
                          min_val: int = 1,
                          max_val: int = 100_000) -> int:
    """Pobiera od użytkownika liczbę całkowitą z zakresu.
    W przypadku błędu powtarza pytanie."""
    while True:
      value = input(prompt)
      if value.isdigit():
        value = int(value)
        if min_val <= value <= max_val:
          return value
      print(f"Błędna wartość. Podaj liczbę całkowitą z zakresu [{min_val}, {max_val}].")

# ---------- dodatkowe funkcje -----------
def get_wagi() -> dict:
    """Konfigurowalny rozkład nukleotydów czyli użytkownik podaje procentowy udział każdego nukleotydu
    (np. A=30, C=20, G=20, T=30). Program waliduje, czy suma wynosi 100%. """

    print("Podaj rozklad nukleotydow, sumarycznie 100%")
    while True:
      try:
        A = float(input("A: "))
        C = float(input("C: "))
        G = float(input("G: "))
        T = float(input("T: "))
        if round(A + C + G + T, 2) == 100:
          return {"A": A, "C": C, "G": G, "T": T}
      except:
        print("Podaj poprawny rozklad, Suma musi wynosic 100%")

def validate_id(prompt: str) -> str:
    """Pobiera od użytkownika nazwę sekwencji i ja waliduje.
    Zakaz znakow bialych."""
    while True:
      value = input(prompt)
      if value and not re.search(r'\s', value):
        return value
      print("Nazwa sekwencji nie może zawierać białych znaków.")

def safe_to_file(fasta: str, filename: str) -> None:
    with open(filename, "w") as file:
      file.write(fasta)

def search_for_motive(motive:str, fasta:str):
  """Wyszukiwanie motywów .. po wygenerowaniu sekwencji program wyszukuje
  w niej wskazany przez użytkownika motyw (np. "ATG") i wypisuje wszystkie
  pozycje wystąpień (indeksowanie od 1, zgodnie z konwencją biologiczną)."""
  onlyFasta = re.sub(r'>.*\n', '', fasta)
  positions = [m.start() + 1 for m in re.finditer(motive, onlyFasta)]
  return positions

def complementarity_and_reverse_complementarity(fasta:str):
  """Sekwencja komplementarna i odwrotnie komplementarna,
  a więc program generuje i wyświetla nić komplementarną
  oraz nić odwrotnie komplementarną (pamiętasz z wykładów co to oznacza?).
  Obie dopisuje jako dodatkowe rekordy do pliku FASTA z odpowiednimi oznaczeniem w ID."""
  lines = fasta.splitlines()
  header = lines[0]
  sequence = "".join(lines[1:])

  id = header.split()[0][1:]
  description = header.split()[1:]
  description = " ".join(description)

  # complementarnosc
  complement_sequence = sequence.maketrans("ATCG", "TAGC")
  complement = sequence.translate(complement_sequence)

  #odwrotnie komplementarnosc
  reversed = complement[::-1]

  #fasta complementarnosc
  fasta_complement = format_fasta(f"{id}_complement", description, complement)

  #fasta reversed
  fasta_reversed = format_fasta(f"{id}_reversed", description, reversed)

  fasta += "\n" + fasta_complement + "\n" + fasta_reversed

  print(f"\nfasta,komplementarnosc i odwrotnie komplementarnosc: \n{fasta}")
  return fasta

def in_siloco_mRNA(fasta:str):
  """Transkrypcja in silico -program generuje sekwencję mRNA (zamiana T na U)
  i zapisuje ją jako odrębny rekord w pliku FASTA."""
  lines = fasta.splitlines()
  header = lines[0]
  sequence = "".join(lines[1:])

  if len(sequence.split(">"))>1:
    sequence = sequence.split(">")[0]

  id = header.split()[0][1:]
  description = header.split()[1:]
  description = " ".join(description)

  inSilico = sequence.replace("T", "U")
  fasta_inSilico = format_fasta(f"{id}_inSilico", description, inSilico)
  fasta += "\n" + fasta_inSilico
  print(f"\nfasta,inSilico: \n{fasta}")
  return fasta

def fasta_file_validation(sciezka:str,dlugoscLiniSekwencji:int = 80, dozwoloneZnaki:str = "ACTG"):
  """Walidacja pliku FASTA
  wymagania:
  - rozpoczyna sie od ">"
  - zawiera ID
  - linijki sekwencji maja 80 lub podana
  - dozwolone znaki w seqwencji"""
  wynik = []

  # sprawdzenie istnienia pliku
  if not os.path.exists(sciezka):
    return "\nPlik nie istnieje"
  if not sciezka.endswith(".fasta"):
    return "\nNie podano pliku .fasta"

  with open(sciezka, "r") as file:
    lines = file.read().splitlines()

  if not lines:
    return "\nPlik jest pusty"

  # sprawdzenie czy PIERWSZY znak to > jest niepotrzebne, bo inaczej sie nie podzieli
  if not lines[0].startswith(">"):
    wynik.append("\nPierwszy znak nie jest >, niemozliwa dalsza analiza")
  records = "\n".join(lines).split(">")
  records = [record for record in records if record] #usuniecie pustego pierwszego wyniku

  for id,record in enumerate(records, start = 1):
    fasta = record.splitlines()
    header = fasta[0]

    # sprawdzenie czy zawiera ID
    parts = header.strip().split()
    if not parts or header.startswith(" "):
      wynik.append(f"- Brak ID-Rekord {id}")

    sequence = "".join(fasta[1:])
    #sprawdzenie czy sekwencja wgl istnieje
    if not sequence:
      wynik.append(f"- Brak sekwencji-Rekord {id}")

    # sprawdzenie dlugosci linijek
    for i,line in enumerate(fasta[1:], start = 1):
      if len(line) > dlugoscLiniSekwencji:
        wynik.append(f"- Dlugosc lini {i} jest wieksza niz {dlugoscLiniSekwencji}, dlugosc lini: {len(line)}-Rekord {id}")

    # sprawdzenie dozwolonych znakow sekwencji
    niedozwoloneZSekwencji = set(sequence.upper()) - set(dozwoloneZnaki)

    if niedozwoloneZSekwencji:
      wynik.append(f"- Niedozwolone znaki w sekwencji+{niedozwoloneZSekwencji}-Rekord {id}")
  if len(wynik)>0:
    return f"w pliku {sciezka}:\n{"\n".join(lines)}\nWynik walidacji:\n"+"\n".join(wynik)
  else:
    return f"Wszystko poprawnie w pliku {sciezka}:\n{"\n".join(lines)}"

def main():
    length = validate_positive_int("Podaj długość sekwencji: ")
    id_seq = validate_id("Podaj ID sekwencji: ")
    description = input("Podaj opis sekwencji: ")
    imie = input("Podaj swoje imię: ").lower()

    # ---------- funkcjonalnosc dodatkowa 2 ---------------
    while True:
      value = input("Czy chcesz wprowadzić własny rozkład nukleotydów? (t/n): ").lower()
      if value == "t":
        wagi = get_wagi()
        print("==wybrano wagi (funkcjonalnosc 2)==")
        break
      elif value == "n":
        wagi = None
        break
      print("Niepoprawna wartość. Podaj 't' lub 'n'.")
    # -----------------------------------------------------

    # sequence = generate_sequence(length)
    sequence = generate_sequence(length, wagi)
    print(sequence)
    sequence_with_name = insert_name(sequence, imie)
    print(sequence_with_name)

    fasta = format_fasta(id_seq, description, sequence)
    print(f"\nfasta:\n{fasta}")

    if imie:
      fasta_with_name = format_fasta(id_seq, description, sequence_with_name)
      print(f"\nfasta_with_name:\n{fasta_with_name}")

    print(f"Statystyki:{calculate_stats(sequence)}")



    # ---------- funkcjonalnosc dodatkowa 3 ---------------
    while True:
      value = input("\nCzy chcesz wyszukać motyw? (t/n): ").lower()
      if value == "t":
        print("==wybrano wyszukiwanie motywów (funkcjonalnosc 3)==")
        while True:
          motive = input("Podaj motyw: ").upper()
          if motive and not re.search(r'\s', motive):
            print(f"Znaleziono pozycje: {search_for_motive(motive, sequence)}")
            break
          else:
            print("Motive nie może zawierać białych znaków.")
        break
      elif value == "n":
        break

    # ---------- funkcjonalnosc dodatkowa 4 ---------------
    while True:
      value = input("\nCzy chcesz dodac do pliku FASTA nic komplementarna i odwrotnie komplementarna? (t/n): ").lower()
      if value == "t":
        print("== wybrano komplementarnie i odwrotnie komplementarnie (funkcjonalnosc 4)==")
        fasta = complementarity_and_reverse_complementarity(fasta)
        break
      elif value == "n":
        break

    # ---------- funkcjonalnosc dodatkowa 5 ---------------
    while True:
      value = input("\nCzy chcesz dodac do pliku FASTA transktypcje in silico (DNA->mRNA)? (t/n)").lower()
      if value == "t":
        print("== wybrano transktypcje in silico (DNA->mRNA) (funkcjonalnosc 5)==")
        fasta = in_siloco_mRNA(fasta)
        break
      elif value == "n":
        break

    # --------- zapis do pliku ---------
    safe_to_file(fasta, f"{id_seq}.fasta")

    # ---------- funkcjonalnosc dodatkowa 11 --------------
    while True:
      value = input("\nCzy chcesz wgrac plik FASTA do zwalidowania? (t/n): ").lower()
      if value == "t":
        print("==wybrano walidacje pliku FASTA (funkcjonalnosc 11)==")
        while True:
          sciezka = input("Podaj sciezke do pliku: ")
          if sciezka and not re.search(r'\s', sciezka):
            print(fasta_file_validation(sciezka))
            break
          else:
            print("Sciezka nie może zawierać białych znaków.")
        break
      elif value == "n":
        break


# Plik musi kończyć się konstrukcją:
if __name__ == "__main__":
    main()


Podaj długość sekwencji: 100
Podaj ID sekwencji: seq`
Podaj opis sekwencji: 
Podaj swoje imię: 
Czy chcesz wprowadzić własny rozkład nukleotydów? (t/n): n
TTCAGCCGGGCTCAAGTGACCAGTTGGGATCAGCCGCCGACTTTATTACTCTACGTGAGGTGCGGAGGATCTAAAGAGAGACAAAATACCAGATCATGGA
TTCAGCCGGGCTCAAGTGACCAGTTGGGATCAGCCGCCGACTTTATTACTCTACGTGAGGTGCGGAGGATCTAAAGAGAGACAAAATACCAGATCATGGA

fasta:
>seq` 
TTCAGCCGGGCTCAAGTGACCAGTTGGGATCAGCCGCCGACTTTATTACTCTACGTGAGGTGCGGAGGATCTAAAGAGAG
ACAAAATACCAGATCATGGA
Statystyki:{'A': 29.0, 'C': 22.0, 'G': 28.0, 'T': 21.0, 'GC_ratio': 50.0}

Czy chcesz wyszukać motyw? (t/n): n

Czy chcesz dodac do pliku FASTA nic komplementarna i odwrotnie komplementarna? (t/n): n

Czy chcesz dodac do pliku FASTA transktypcje in silico (DNA->mRNA)? (t/n)n

Czy chcesz wgrac plik FASTA do zwalidowania? (t/n): t
==wybrano walidacje pliku FASTA (funkcjonalnosc 11)==
Podaj sciezke do pliku: se1.fasta
Wszystko poprawnie w pliku se1.fasta:
>se1 opisssss
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA